In [28]:
import json
import glob
import re
import pandas as pd
from pathlib import Path
from natsort import natsorted
from format_tex import format_result
import os

In [29]:
RUN_LABEL = 'news999'
DATASETS = os.listdir(Path('../results/') / RUN_LABEL)
DATASET_PATH = Path('../data/classification/news_ideology')
DATASET_INDEX = 'Unnamed: 0'
RESULT_DIRS = [
    Path('../results/') / RUN_LABEL / dataset for dataset in DATASETS
]
LABELS = ['Liberal', 'Neutral', 'Conservative']

In [30]:
results = []
for result_dir in RESULT_DIRS:
    results += glob.glob(str(result_dir / '**/balanced_bertscore/8000_cands*/**/test-1000.json'), recursive=True)
    results += glob.glob(str(result_dir /  '**/random/8000_cands*/**/test-1000.json'), recursive=True)
results = [i for i in results if 'GPT' in i]
results

['../results/news999/NEWSSOURCEIDEOLOGY/test/4_shots/balanced_bertscore/8000_cands-deberta-large-mnli-recall/s0/GPT4/test-1000.json',
 '../results/news999/NEWSSOURCEIDEOLOGY/test/12_shots/balanced_bertscore/8000_cands-deberta-large-mnli-recall/s0/GPT4/test-1000.json',
 '../results/news999/NEWSSOURCEIDEOLOGY/test/8_shots/balanced_bertscore/8000_cands-deberta-large-mnli-recall/s0/GPT4/test-1000.json',
 '../results/news999/NEWSSOURCEIDEOLOGY/test/0_shots/random/8000_cands/s0/GPT4/test-1000.json',
 '../results/news999/NEWSDESCRIPTIONIDEOLOGY/test/4_shots/balanced_bertscore/8000_cands-deberta-large-mnli-recall/s0/GPT4/test-1000.json',
 '../results/news999/NEWSDESCRIPTIONIDEOLOGY/test/8_shots/balanced_bertscore/8000_cands-deberta-large-mnli-recall/s0/GPT4/test-1000.json',
 '../results/news999/NEWSDESCRIPTIONIDEOLOGY/test/0_shots/random/8000_cands/s0/GPT4/test-1000.json',
 '../results/news999/NEWSSOURCEDESCRIPTIONIDEOLOGY/test/0_shots/random/8000_cands/s0/GPT4/test-1000.json',
 '../results/ne

In [31]:
def sanitize_prediction(x):
    if 'neutral' in x.lower():
        return 'Neutral'
    if 'liberal' in x.lower():
        return 'Liberal'
    if 'conservative' in x.lower():
        return 'Conservative'
    raise Exception('Invalid response', x)

In [32]:
llm_names = set()
num_shots_ = set()
datasets = set()

test_set = pd.read_json(DATASET_PATH / 'test_small.json')
test_set = test_set.set_index('article_id')
test_set['true'] = test_set['label'].map(lambda x : LABELS[x])

keys = []

for result in results:
    regex = f'../results/{RUN_LABEL}/(?P<dataset>.*?)/test/(?P<num_shots>[0-9]+)?_shots/.*?/[0-9]+?_cands.*?/s0/(?P<llm_name>.*)?/test-1000.json'
    groups = re.search(
        re.compile(regex),
        result
    )

    llm_name = groups.group('llm_name')
    num_shots = groups.group('num_shots')
    dataset = groups.group('dataset')

    llm_names.add(llm_name)
    num_shots_.add(num_shots)
    datasets.add(dataset)

    print(llm_name, num_shots, dataset)

    with open(result) as f:
        js = json.load(f)

    key = f'{llm_name}_{num_shots}_{dataset}'
    keys.append(key)
    
    rows = []
    for res in js['results']:
        try:
            pred = sanitize_prediction(res['pred'])
            idx = res['article_id']
            rows.append({
                key: pred,
                'idx': idx
            })
        except Exception as e:
            print(e)
            continue
    df = pd.DataFrame(rows).set_index('idx')
    test_set = test_set.merge(df, left_index=True, right_index=True)

keys = natsorted(keys)
test_set = test_set.dropna(subset=keys)

GPT4 4 NEWSSOURCEIDEOLOGY
GPT4 12 NEWSSOURCEIDEOLOGY
GPT4 8 NEWSSOURCEIDEOLOGY
GPT4 0 NEWSSOURCEIDEOLOGY
GPT4 4 NEWSDESCRIPTIONIDEOLOGY
GPT4 8 NEWSDESCRIPTIONIDEOLOGY
GPT4 0 NEWSDESCRIPTIONIDEOLOGY
GPT4 0 NEWSSOURCEDESCRIPTIONIDEOLOGY
GPT4 4 NEWSIDEOLOGY
GPT4 8 NEWSIDEOLOGY
GPT4 0 NEWSIDEOLOGY
('Invalid response', 'Please provide the news article titles you would like classified.')


In [33]:
for dataset in sorted(datasets, key=len):
    print('\\multicolumn{9}{c}{%s}\\\\ \\midrule' % dataset)
    for llm_name in sorted(llm_names):
        for num_shots in natsorted(num_shots_):
            key = f'{llm_name}_{num_shots}_{dataset}'
            if key in test_set.columns:
                print(format_result(llm_name, num_shots, test_set['true'], test_set[key], LABELS), end='\\\\\n')

\multicolumn{9}{c}{NEWSIDEOLOGY}\\ \midrule
GPT4 & 0 & 0.69 & 0.85 & 0.56 & 0.82 & 0.47 & 0.92 & 0.67\\
GPT4 & 4 & 0.76 & 0.75 & 0.72 & 0.81 & 0.72 & 0.77 & 0.78\\
GPT4 & 8 & 0.74 & 0.72 & 0.73 & 0.78 & 0.72 & 0.73 & 0.77\\
\multicolumn{9}{c}{NEWSSOURCEIDEOLOGY}\\ \midrule
GPT4 & 0 & 0.84 & 0.90 & 0.74 & 0.92 & 0.75 & 0.86 & 0.92\\
GPT4 & 4 & 0.88 & 0.87 & 0.84 & 0.92 & 0.88 & 0.79 & 0.96\\
GPT4 & 8 & 0.87 & 0.86 & 0.85 & 0.90 & 0.90 & 0.76 & 0.96\\
GPT4 & 12 & 0.87 & 0.87 & 0.84 & 0.91 & 0.88 & 0.78 & 0.95\\
\multicolumn{9}{c}{NEWSDESCRIPTIONIDEOLOGY}\\ \midrule
GPT4 & 0 & 0.79 & 0.84 & 0.67 & 0.92 & 0.68 & 0.89 & 0.80\\
GPT4 & 4 & 0.81 & 0.80 & 0.77 & 0.87 & 0.79 & 0.78 & 0.87\\
GPT4 & 8 & 0.82 & 0.80 & 0.77 & 0.88 & 0.79 & 0.80 & 0.87\\
\multicolumn{9}{c}{NEWSSOURCEDESCRIPTIONIDEOLOGY}\\ \midrule
GPT4 & 0 & 0.85 & 0.88 & 0.75 & 0.95 & 0.78 & 0.86 & 0.91\\
GPT4 & 8 & 0.82 & 0.80 & 0.77 & 0.88 & 0.79 & 0.80 & 0.87\\
\multicolumn{9}{c}{NEWSSOURCEDESCRIPTIONIDEOLOGY}\\ \midrule
GPT4 & 0

In [34]:
last_model = None
batch = []
for dataset in sorted(datasets, key=len):
    for model in sorted(llm_names):
        for num_shots in sorted(num_shots_):
            if last_model and model != last_model:
                print('\\multirow{%s}{*}{%s}\n' % (len(batch), last_model), end='')
                print(' \\\\ \n'.join(batch), end='')
                print(' \\\\ \n \\midrule')
                batch = []
            batch.append(format_result('', num_shots, test_set['true'], test_set[key], labels=LABELS))
            last_model = model

if len(batch) > 0:
    print('\\multirow{%s}{*}{%s}\n' % (len(batch), last_model), end='')
    print(' \\\\ \n'.join(batch), end='')
    print(' \\\\ \n')
    batch = []

KeyError: 'GPT4_12_NEWSSOURCEDESCRIPTIONIDEOLOGY'